# KMeans Clustering — Interactive Demo

**scikit-learn | ipywidgets | matplotlib**

---

## What is Clustering?

Clustering is an **unsupervised machine learning** technique that groups similar data points together — without using any labels.

## What is KMeans?

**KMeans** is one of the most popular clustering algorithms. It works by:

1. Randomly placing **K centroids** in the feature space
2. Assigning each data point to its **nearest centroid**
3. Moving each centroid to the **mean** of its assigned points
4. Repeating steps 2–3 until the centroids stop moving (convergence)

> **Your goal:** Use the interactive slider below to change the number of clusters (K) and observe how the algorithm partitions the data differently.

---

## 📦 Step 1 — Install & Import Libraries

Run this cell first to make sure all dependencies are available.

In [ ]:
# Install ipywidgets if not already installed
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ipywidgets', '--quiet'])

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, HBox, VBox, Output
from IPython.display import display, clear_output

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('✅ All libraries loaded successfully!')

---

## 🗄️ Step 2 — Generate a Synthetic Dataset

We use `make_blobs` from scikit-learn to create a 2D dataset that is easy to visualize.

You can change the parameters below to experiment:
- `n_samples` — total number of data points
- `true_k` — the **actual** number of clusters hidden in the data
- `cluster_std` — how spread out (noisy) each cluster is

In [ ]:
# ── Dataset Parameters ──────────────────────────────────────────
n_samples  = 500   # Total number of data points
true_k     = 6     # True number of clusters in the data
cluster_std = 2.0 # Spread of each cluster (higher = more overlap)
# ────────────────────────────────────────────────────────────────

X_raw, y_true = make_blobs(
    n_samples=n_samples,
    centers=true_k,
    cluster_std=cluster_std,
    random_state=RANDOM_STATE
)

# Standardize features (good practice before clustering)
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

# Quick preview plot
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X[:, 0], X[:, 1], c='steelblue', s=30, alpha=0.6, edgecolors='white', linewidths=0.4)
ax.set_title(f'Raw Dataset — {n_samples} points (labels hidden)', fontsize=13, fontweight='bold')
ax.set_xlabel('Feature 1 (standardized)')
ax.set_ylabel('Feature 2 (standardized)')
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

print(f'Dataset shape: {X.shape}  |  True clusters hidden: {true_k}')

---

## 🎨 Step 3 — Interactive KMeans Visualizer

Use the **slider** to choose K (number of clusters). The plot will update to show:

- 🎨 **Colored points** — each color is an assigned cluster
- ⭐ **Star markers** — cluster centroids
- 🗺️ **Background shading** — decision regions (which area belongs to which cluster)
- 📊 **Silhouette score** — a quality metric (closer to 1.0 = better-defined clusters)

> 💡 **Tip:** Try K = 4, which matches the true number of clusters. Then try K = 2 or K = 7 to see what happens!

In [ ]:
# ── Color palette ────────────────────────────────────────────────
PALETTE = [
    '#E63946', '#2196F3', '#4CAF50', '#FF9800',
    '#9C27B0', '#00BCD4', '#FF5722', '#607D8B',
    '#795548', '#8BC34A'
]

def run_kmeans_and_plot(k):
    """Fit KMeans with k clusters and display the full visualization."""

    # ── Fit KMeans ───────────────────────────────────────────────
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X)
    centroids = km.cluster_centers_

    # ── Silhouette score (needs k >= 2) ──────────────────────────
    sil = silhouette_score(X, labels) if k >= 2 else float('nan')

    # ── Build mesh for decision boundary ────────────────────────
    h = 0.05
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = km.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    colors = PALETTE[:k]
    cmap_bg = ListedColormap([c + '33' for c in colors])  # translucent bg
    cmap_pt = ListedColormap(colors)

    # ── Figure ───────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle(f'KMeans Clustering  |  K = {k}  |  Silhouette Score = {sil:.3f}',
                 fontsize=14, fontweight='bold', y=1.01)

    # ── Left: Cluster plot ───────────────────────────────────────
    ax = axes[0]
    ax.contourf(xx, yy, Z, cmap=cmap_bg, alpha=0.9)
    ax.contour(xx, yy, Z, colors='white', linewidths=0.8, alpha=0.5)

    for cluster_id in range(k):
        mask = labels == cluster_id
        ax.scatter(X[mask, 0], X[mask, 1],
                   c=colors[cluster_id], s=35, alpha=0.85,
                   edgecolors='white', linewidths=0.4,
                   label=f'Cluster {cluster_id + 1}')

    ax.scatter(centroids[:, 0], centroids[:, 1],
               c='white', s=250, marker='*', zorder=5,
               edgecolors='black', linewidths=0.8, label='Centroids')

    ax.set_title('Cluster Assignments & Decision Regions', fontsize=12)
    ax.set_xlabel('Feature 1 (standardized)')
    ax.set_ylabel('Feature 2 (standardized)')
    ax.legend(loc='upper right', fontsize=8, framealpha=0.85)
    ax.grid(True, linestyle='--', alpha=0.25)

    # ── Right: Silhouette plot ───────────────────────────────────
    ax2 = axes[1]
    if k >= 2:
        sil_vals = silhouette_samples(X, labels)
        y_lower = 10
        for i in range(k):
            ith_vals = np.sort(sil_vals[labels == i])
            size_i = ith_vals.shape[0]
            y_upper = y_lower + size_i
            ax2.fill_betweenx(np.arange(y_lower, y_upper),
                              0, ith_vals, facecolor=colors[i], alpha=0.8)
            ax2.text(-0.05, y_lower + 0.5 * size_i, str(i + 1), fontsize=9)
            y_lower = y_upper + 10

        ax2.axvline(x=sil, color='red', linestyle='--', linewidth=1.5,
                    label=f'Avg = {sil:.3f}')
        ax2.set_xlabel('Silhouette Coefficient')
        ax2.set_ylabel('Cluster')
        ax2.set_title('Silhouette Plot (width = cluster quality)', fontsize=12)
        ax2.set_yticks([])
        ax2.set_xlim([-0.3, 1.0])
        ax2.legend(fontsize=9)
        ax2.grid(True, linestyle='--', alpha=0.25)
    else:
        ax2.text(0.5, 0.5, 'Silhouette requires K ≥ 2',
                 ha='center', va='center', fontsize=12, transform=ax2.transAxes)
        ax2.set_axis_off()

    plt.tight_layout()
    plt.show()

    # ── Stats summary ────────────────────────────────────────────
    print(f'\n📊 Summary for K = {k}')
    print(f'   Inertia (within-cluster sum of squares): {km.inertia_:.2f}')
    print(f'   Silhouette Score:                        {sil:.4f}  (range: -1 to 1, higher = better)')
    print(f'   Iterations to converge:                  {km.n_iter_}')
    sizes = np.bincount(labels)
    for i, s in enumerate(sizes):
        print(f'   Cluster {i+1}: {s} points')


# ── Widget UI ────────────────────────────────────────────────────
k_slider = widgets.IntSlider(
    value=2,
    min=1, max=10, step=1,
    description='K (clusters):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='420px')
)

out = Output()

def on_slider_change(change):
    with out:
        clear_output(wait=True)
        run_kmeans_and_plot(change['new'])

k_slider.observe(on_slider_change, names='value')

display(VBox([
    widgets.HTML('<h3 style="margin-bottom:4px">🎛️ Adjust Number of Clusters</h3>'),
    k_slider,
    out
]))

# Trigger initial render
with out:
    run_kmeans_and_plot(k_slider.value)

---

## 📈 Step 4 — The Elbow Method

How do we **choose the best K**? One popular technique is the **Elbow Method**:

- We run KMeans for K = 1 to 10
- We record the **inertia** (sum of squared distances from each point to its centroid)
- We look for the **"elbow"** — the point where adding more clusters stops giving much improvement

> 🔑 The elbow point is a good candidate for the optimal K.

In [ ]:
k_range = range(1, 11)
inertias = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    km.fit(X)
    inertias.append(km.inertia_)
    if k >= 2:
        sil_scores.append(silhouette_score(X, km.labels_))
    else:
        sil_scores.append(None)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow curve
ax = axes[0]
ax.plot(list(k_range), inertias, 'o-', color='#2196F3', linewidth=2.5, markersize=7)
ax.axvline(x=true_k, color='red', linestyle='--', linewidth=1.5,
           label=f'True K = {true_k}')
ax.set_title('Elbow Method — Inertia vs K', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Clusters (K)')
ax.set_ylabel('Inertia')
ax.set_xticks(list(k_range))
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)

# Silhouette curve
ax2 = axes[1]
valid_k = [k for k in k_range if k >= 2]
valid_sil = [s for s in sil_scores if s is not None]
ax2.plot(valid_k, valid_sil, 'o-', color='#4CAF50', linewidth=2.5, markersize=7)
ax2.axvline(x=true_k, color='red', linestyle='--', linewidth=1.5,
            label=f'True K = {true_k}')
ax2.set_title('Silhouette Score vs K  (higher = better)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Number of Clusters (K)')
ax2.set_ylabel('Silhouette Score')
ax2.set_xticks(list(k_range))
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

best_k = valid_k[np.argmax(valid_sil)]
print(f'\n🏆 Best K by silhouette score: K = {best_k}  (true K = {true_k})')

---

## 🔬 Step 5 — Watch KMeans Converge (Step-by-Step)

This section lets you **watch the algorithm converge** one iteration at a time.

Each step shows how centroids move toward the cluster centers. Use the **dropdown** to choose K, then click **▶ Next Step**.

In [ ]:
# ── Step-by-step convergence viewer ─────────────────────────────

state = {'centroids': None, 'labels': None, 'step': 0, 'k': 3}

def init_convergence(k):
    """Initialise centroids randomly for a fresh convergence demo."""
    idx = np.random.choice(len(X), k, replace=False)
    state['centroids'] = X[idx].copy()
    state['labels'] = np.zeros(len(X), dtype=int)
    state['step'] = 0
    state['k'] = k

def assign_labels(centroids):
    dists = np.linalg.norm(X[:, None] - centroids[None, :], axis=2)
    return np.argmin(dists, axis=1)

def update_centroids(labels, k):
    new_c = np.array([X[labels == i].mean(axis=0) if (labels == i).any()
                      else state['centroids'][i]
                      for i in range(k)])
    return new_c

def plot_convergence_step():
    k = state['k']
    colors = PALETTE[:k]
    fig, ax = plt.subplots(figsize=(8, 6))

    for i in range(k):
        mask = state['labels'] == i
        ax.scatter(X[mask, 0], X[mask, 1],
                   c=colors[i], s=30, alpha=0.6, edgecolors='white', linewidths=0.3)

    ax.scatter(state['centroids'][:, 0], state['centroids'][:, 1],
               c='white', s=300, marker='*', zorder=5,
               edgecolors='black', linewidths=1.2)
    for i, c in enumerate(state['centroids']):
        ax.annotate(f'C{i+1}', c, textcoords='offset points',
                    xytext=(8, 4), fontsize=10, fontweight='bold')

    ax.set_title(f'Convergence Demo  |  K = {k}  |  Step {state["step"]}',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f'  Step {state["step"]}: centroids updated.')


# Widgets
k_drop = widgets.Dropdown(
    options=list(range(2, 9)), value=3,
    description='K:', layout=widgets.Layout(width='150px')
)
btn_init = widgets.Button(description='🔄 Restart', button_style='warning',
                          layout=widgets.Layout(width='130px'))
btn_step = widgets.Button(description='▶ Next Step', button_style='success',
                          layout=widgets.Layout(width='130px'))
step_out = Output()

def on_restart(b):
    init_convergence(k_drop.value)
    with step_out:
        clear_output(wait=True)
        print(f'🔄 Restarted with K = {k_drop.value}. Click ▶ Next Step to iterate.')
        plot_convergence_step()

def on_next_step(b):
    if state['centroids'] is None:
        init_convergence(k_drop.value)

    old_centroids = state['centroids'].copy()
    state['labels'] = assign_labels(state['centroids'])
    state['centroids'] = update_centroids(state['labels'], state['k'])
    state['step'] += 1

    moved = np.linalg.norm(state['centroids'] - old_centroids)
    with step_out:
        clear_output(wait=True)
        plot_convergence_step()
        if moved < 1e-6:
            print('✅ Converged! Centroids are no longer moving.')
        else:
            print(f'   Total centroid movement: {moved:.6f}')

btn_init.on_click(on_restart)
btn_step.on_click(on_next_step)

display(VBox([
    widgets.HTML('<h3>🔬 Step-by-Step Convergence Viewer</h3>'),
    HBox([k_drop, btn_init, btn_step]),
    step_out
]))

# Init on load
init_convergence(k_drop.value)
with step_out:
    print(f'Ready. K = {k_drop.value}. Click ▶ Next Step to begin.')
    plot_convergence_step()

---

## 🧠 Key Takeaways

| Concept | Summary |
|---|---|
| **KMeans** | Partitions data into K clusters by minimizing inertia |
| **K (number of clusters)** | Must be chosen by the user — the algorithm doesn't find it automatically |
| **Inertia** | Sum of squared distances from each point to its centroid — lower is better, but always decreases with more K |
| **Silhouette Score** | Measures how similar a point is to its own cluster vs others — higher is better (range: -1 to 1) |
| **Elbow Method** | Find the K where inertia starts to decrease more slowly |
| **Convergence** | Algorithm stops when centroids no longer move significantly |

---

## 🚀 Further Exploration

1. Go back to **Step 2** and change `cluster_std` to `2.0` — clusters overlap more. Can KMeans still find them?
2. Change `true_k` to `6` — does the Elbow Method still identify the correct K?
3. Try `n_samples = 1000` for a larger dataset. Does convergence take more steps?
4. Explore other clustering algorithms: `DBSCAN`, `AgglomerativeClustering`, `GaussianMixture` from scikit-learn.